In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [2]:
df = pd.read_csv("../data/raw/payflow_transactions.csv")
print(df.shape)
df.head(3)

(150000, 20)


,transaction_id,timestamp,user_id,merchant_id,amount,currency,merchant_category,payment_method,entry_mode,account_age_days,user_home_country,kyc_verified,device_type,is_new_device,ip_country,is_vpn_or_proxy,distance_from_home_km,failed_attempts_1h,is_international,is_fraud
0,TX_000001,2026-05-01 00:04:59,USR_04153,MER_375,154.51,SAR,electronics,credit_card,online_no_3ds,452.0,SA,0,ios,0,SA,0,24.10,0,0,0
1,TX_000002,2026-05-01 00:23:03,USR_07866,MER_261,577.81,SAR,electronics,credit_card,chip,766.0,SA,1,pos,0,NaN,0,45.27,1,0,0
2,TX_000003,2026-05-01 00:23:58,USR_09294,MER_465,528.23,EGP,groceries,credit_card,contactless,228.0,EG,1,pos,0,NaN,0,19.25,0,0,0


In [3]:
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values(["timestamp", "transaction_id"]).reset_index(drop=True)

In [4]:
fx_to_usd = {
     "USD": 1.0,
         "EUR": 0.92,
         "GBP": 0.79,
         "SAR": 3.75,
         "EGP": 48.5,
}

df["amount_usd"] = (df["amount"] / df["currency"].map(fx_to_usd)).round(2)

In [5]:
df["hour_of_day"] = df["timestamp"].dt.hour
df["is_night"] = df["timestamp"].dt.hour.between(1, 5).astype(int)
df["ip_country_mismatch"] = ((df["ip_country"].notna()) & (df["ip_country"] != df["user_home_country"])).astype(int)

df = df.sort_values(["user_id", "timestamp", "transaction_id"]).reset_index(drop=True)

In [6]:
for col_name, window in [("user_tx_count_1h", "1h"), ("user_tx_count_24h", "24h")]:
        df[col_name] = (
            df.groupby("user_id", sort=False)
              .rolling(window, on="timestamp", closed="left")["amount_usd"]
              .count()
              .fillna(0)
              .astype(int)
              .to_numpy()
        )
df["time_since_last_tx_sec"] = ( df.groupby("user_id", sort=False)["timestamp"].diff().dt.total_seconds())

In [7]:
df["user_avg_amount_usd"] = (
    df.groupby("user_id", sort=False)["amount_usd"]
      .transform(lambda s: s.shift(1).expanding().mean())
)
df["amount_to_user_avg_ratio"] = df["amount_usd"] / df["user_avg_amount_usd"]

In [8]:
start = df["timestamp"].min().normalize()
train_end = start + pd.Timedelta(days=120)
global_prior = df.loc[df["timestamp"] < train_end, "is_fraud"].mean()

df = df.sort_values(["timestamp", "transaction_id"]).reset_index(drop=True)

In [9]:
m_hist = df[["merchant_id", "timestamp", "is_fraud"]].copy()
m_hist["m_tx_cum"] = m_hist.groupby("merchant_id").cumcount() + 1
m_hist["m_fraud_cum"] = m_hist.groupby("merchant_id")["is_fraud"].cumsum()
m_hist = m_hist.drop_duplicates(subset=["merchant_id", "timestamp"], keep="last")
df["ts_minus_7d"] = df["timestamp"] - pd.Timedelta(days=7)

In [10]:
df = pd.merge_asof(
    df,
    m_hist[["merchant_id", "timestamp", "m_tx_cum", "m_fraud_cum"]].rename(
        columns={"timestamp": "hist_ts"}
    ),
    left_on="ts_minus_7d",
    right_on="hist_ts",
    by="merchant_id",
    direction="backward",
)

In [11]:
m = 20
past_tx = df["m_tx_cum"].fillna(0)
past_fraud = df["m_fraud_cum"].fillna(0)

df["merchant_fraud_rate_7d_lag"] = (past_fraud + m * global_prior) / (past_tx + m)

df = df.drop(columns=["ts_minus_7d", "hist_ts", "m_tx_cum", "m_fraud_cum"])

In [12]:
val_end = train_end + pd.Timedelta(days=30)
test_end = val_end + pd.Timedelta(days=30)

train = df[df['timestamp'] < train_end].copy()
val = df[(df['timestamp'] >= train_end) & (df['timestamp'] < val_end)].copy()
test = df[(df['timestamp'] >= val_end) & (df['timestamp'] < test_end)].copy()

for name, part in [('train', train), ('val', val), ('test', test)]:
    print(f"{name}: {len(part):>7} rows | "
          f"{part['timestamp'].min().date()} -> {part['timestamp'].max().date()} | "
          f"fraud rate: {part['is_fraud'].mean():.2%}")

train:  100177 rows | 2026-05-01 -> 2026-08-28 | fraud rate: 2.05%
val:   24720 rows | 2026-08-29 -> 2026-09-27 | fraud rate: 1.58%
test:   25103 rows | 2026-09-28 -> 2026-10-27 | fraud rate: 2.26%


In [13]:
train = train.drop(columns=["timestamp", "transaction_id", "user_id", "merchant_id", "amount", "user_avg_amount_usd"])
val = val.drop(columns=["timestamp", "transaction_id", "user_id", "merchant_id", "amount", "user_avg_amount_usd"])
test = test.drop(columns=["timestamp", "transaction_id", "user_id", "merchant_id", "amount", "user_avg_amount_usd"])

In [14]:
x_train = train.drop(columns=["is_fraud"])
y_train = train["is_fraud"]

In [15]:
x_val = val.drop(columns=["is_fraud"])
y_val = val["is_fraud"]

In [16]:
NUMERIC_COLS: list[str] = [
     "account_age_days",
     "failed_attempts_1h",
     "hour_of_day",
     "user_tx_count_1h",
     "user_tx_count_24h",
     "merchant_fraud_rate_7d_lag",
]

In [17]:
LOG_NUMERIC_COLS: list[str] = [
    "amount_usd",
    "distance_from_home_km",
    "time_since_last_tx_sec",
    "amount_to_user_avg_ratio",
]

In [18]:
BINARY_COLS: list[str] = [
     "kyc_verified",
     "is_new_device",
     "is_vpn_or_proxy",
     "is_international",
     "is_night",
     "ip_country_mismatch",
]

In [19]:
CATEGORICAL_COLS: list[str] = [
     "currency",
     "merchant_category",
     "payment_method",
     "entry_mode",
     "user_home_country",
     "device_type",
     "ip_country",
]


In [20]:
num_pipeline = Pipeline(
     steps=[
          ("imputer", SimpleImputer(strategy="median")),
          ("scaler", StandardScaler())
     ]
)

In [21]:
log_num_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
        ("scaler", StandardScaler()),
    ]
)

In [22]:
cat_pipeline = Pipeline(
     steps=[
          ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
          ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
     ]
)

In [23]:
preprocessor = ColumnTransformer(
     transformers= [
          ("num", num_pipeline, NUMERIC_COLS),
          ("log_num", log_num_pipeline, LOG_NUMERIC_COLS),
          ("cat", cat_pipeline, CATEGORICAL_COLS),
          ("bin", "passthrough", BINARY_COLS)
     ]
)

In [24]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

models = [
     ("logistic", LogisticRegression(
          penalty="l2",
          C=1.0,
          solver="liblinear",
          class_weight="balanced",
          random_state=42,
          n_jobs=1
     )),
     ("xgboost",
      XGBClassifier(
           n_estimators=400,
           learning_rate=0.03,
           max_depth=4,
           min_child_weight=5,
           subsample=0.8,
           colsample_bytree=0.8,
           reg_alpha=0.5,
           reg_lambda=2.0,
           eval_metric="logloss",
           scale_pos_weight=scale_pos_weight,
           random_state=42,
           n_jobs=1,
     )),
     ("lightgbm", LGBMClassifier(
          n_estimators=400,
          learning_rate=0.03,
          max_depth=4,
          num_leaves=15, 
          min_child_samples=20,
          subsample=0.8,
          subsample_freq=1,
          colsample_bytree=0.8,
          reg_alpha=0.5,
          reg_lambda=2.0,
          scale_pos_weight=scale_pos_weight,
          random_state=42,
          n_jobs=1,
          verbose=-1,              
     ))
]

In [25]:
val_results = []

for name, model in models:
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model),
    ])
    pipe.fit(x_train, y_train)

    train_probs = pipe.predict_proba(x_train)[:, 1]
    val_probs = pipe.predict_proba(x_val)[:, 1]

    val_results.append({
        "Model": name,
        "Train PR-AUC": round(average_precision_score(y_train, train_probs), 4),
        "Val PR-AUC": round(average_precision_score(y_val, val_probs), 4),
        "Train ROC-AUC": round(roc_auc_score(y_train, train_probs), 4),
        "Val ROC-AUC": round(roc_auc_score(y_val, val_probs), 4),
    })

pd.DataFrame(val_results).sort_values("Val PR-AUC", ascending=False)

,Model,Train PR-AUC,Val PR-AUC,Train ROC-AUC,Val ROC-AUC
2,lightgbm,0.5827,0.4167,0.9574,0.9096
1,xgboost,0.5840,0.4160,0.9573,0.9084
0,logistic,0.4942,0.4056,0.9260,0.9117


In [26]:
lgb_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", models[2][1]),  # lightgbm
])
lgb_pipe.fit(x_train, y_train)

feature_names = lgb_pipe.named_steps["preprocessor"].get_feature_names_out()
importances = lgb_pipe.named_steps["model"].feature_importances_

pd.Series(importances, index=feature_names).sort_values(ascending=False).head(15)

log_num__distance_from_home_km       656
log_num__amount_usd                  562
num__account_age_days                559
log_num__time_since_last_tx_sec      492
log_num__amount_to_user_avg_ratio    442
num__merchant_fraud_rate_7d_lag      438
num__hour_of_day                     371
num__failed_attempts_1h              211
bin__is_new_device                   123
cat__entry_mode_online_no_3ds        119
bin__kyc_verified                    113
bin__is_international                112
bin__is_vpn_or_proxy                 105
bin__ip_country_mismatch              95
cat__merchant_category_groceries      91
dtype: int32